# MLOps Lab 4 — Wine Quality Classification

**Modifications from original lab:**
- **Dataset**: UCI Wine Quality dataset (red wine, 1599 samples, 11 features) instead of Dermatology
- **Task**: Binary classification — predict whether wine quality is *good* (score ≥ 7) or *not good*
- **Models**: XGBoost + Random Forest comparison (original had only XGBoost)
- **Extra W&B logging**: Feature importance chart, ROC-AUC curve, PR curve, and per-run comparison table
- **Hyperparameter sweep**: W&B sweep over `max_depth` and `eta` for XGBoost


## 1. Setup & Imports

In [ ]:
# Install dependencies if needed
# !pip install wandb xgboost scikit-learn pandas matplotlib seaborn --quiet

In [ ]:
import os
import wandb
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import StandardScaler

print("All imports successful!")

In [ ]:
wandb.login()

## 2. Load & Explore the Dataset

Using the **UCI Red Wine Quality** dataset. Each row is a wine sample described by 11 physicochemical features. The target `quality` is an integer 3–8; we binarize it: quality ≥ 7 → **good (1)**, otherwise → **not good (0)**.

In [ ]:
# Download dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=';')

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution (raw quality scores):")
print(df['quality'].value_counts().sort_index())
df.head()

In [ ]:
# Binarize the target: good wine = quality >= 7
df['label'] = (df['quality'] >= 7).astype(int)

print("Binary class distribution:")
print(df['label'].value_counts())
print(f"\nClass balance: {df['label'].mean():.1%} good wines")

# Feature matrix and target
FEATURES = [c for c in df.columns if c not in ('quality', 'label')]
X = df[FEATURES].values
y = df['label'].values

print(f"\nFeatures ({len(FEATURES)}): {FEATURES}")

In [ ]:
# Correlation heatmap (logged to W&B later)
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(df[FEATURES + ['label']].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Feature Correlation Heatmap — Red Wine Quality')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120)
plt.show()
print("Heatmap saved.")

## 3. Train / Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

## 4. Helper — Log Rich Metrics to W&B

In [ ]:
def log_model_metrics(run, model_name, y_true, y_pred, y_prob, feature_names, importances=None):
    """Log accuracy, ROC-AUC, PR curve, confusion matrix, and feature importance to W&B."""

    acc    = accuracy_score(y_true, y_pred)
    roc    = roc_auc_score(y_true, y_prob)
    ap     = average_precision_score(y_true, y_prob)

    run.summary[f"{model_name}/accuracy"]         = round(acc, 4)
    run.summary[f"{model_name}/roc_auc"]          = round(roc, 4)
    run.summary[f"{model_name}/avg_precision"]    = round(ap,  4)

    print(f"\n{'='*40}")
    print(f"  {model_name}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  ROC-AUC  : {roc:.4f}")
    print(f"  Avg Prec : {ap:.4f}")
    print(f"{'='*40}")
    print(classification_report(y_true, y_pred, target_names=['Not Good', 'Good']))

    # --- Confusion matrix ---
    wandb.sklearn.plot_confusion_matrix(y_true, y_pred, ['Not Good', 'Good'])

    # --- ROC curve ---
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots()
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={roc:.3f})")
    ax.plot([0,1],[0,1],'k--')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve'); ax.legend()
    run.log({f"{model_name}/roc_curve": wandb.Image(fig)})
    plt.close(fig)

    # --- PR curve ---
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    fig2, ax2 = plt.subplots()
    ax2.plot(rec, prec, label=f"{model_name} (AP={ap:.3f})")
    ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
    ax2.set_title('Precision-Recall Curve'); ax2.legend()
    run.log({f"{model_name}/pr_curve": wandb.Image(fig2)})
    plt.close(fig2)

    # --- Feature importance ---
    if importances is not None:
        fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
        fi_df = fi_df.sort_values('importance', ascending=True)
        fig3, ax3 = plt.subplots(figsize=(8, 5))
        ax3.barh(fi_df['feature'], fi_df['importance'], color='steelblue')
        ax3.set_title(f'{model_name} — Feature Importance')
        ax3.set_xlabel('Importance Score')
        plt.tight_layout()
        run.log({f"{model_name}/feature_importance": wandb.Image(fig3)})
        plt.close(fig3)

print("Helper function defined.")

## 5. Model 1 — XGBoost Classifier

In [ ]:
xgb_params = {
    'objective':  'binary:logistic',
    'eta':        0.1,
    'max_depth':  6,
    'subsample':  0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'logloss',
    'seed':       42,
}

run_xgb = wandb.init(
    project="Lab1-wine-quality-mlops",
    name="xgboost-wine",
    config=xgb_params
)

# Log dataset artifact
run_xgb.log({"correlation_heatmap": wandb.Image('correlation_heatmap.png')})

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=FEATURES)
watchlist = [(dtrain, 'train'), (dtest, 'eval')]

bst = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    early_stopping_rounds=15,
    verbose_eval=10,
    callbacks=[wandb.xgboost.WandbCallback()]
)

y_prob_xgb = bst.predict(dtest)
y_pred_xgb = (y_prob_xgb >= 0.5).astype(int)

# Feature importance
fi_scores = bst.get_score(importance_type='gain')
fi_vals   = np.array([fi_scores.get(f, 0) for f in FEATURES])

log_model_metrics(run_xgb, 'XGBoost', y_test, y_pred_xgb, y_prob_xgb, FEATURES, fi_vals)

run_xgb.finish()
print("\nXGBoost run complete!")

## 6. Model 2 — Random Forest Classifier

Added as a comparison model not present in the original lab.

In [ ]:
rf_params = {
    'n_estimators': 300,
    'max_depth':    10,
    'min_samples_split': 5,
    'class_weight': 'balanced',   # handles class imbalance
    'random_state': 42,
    'n_jobs':       -1
}

run_rf = wandb.init(
    project="Lab1-wine-quality-mlops",
    name="random-forest-wine",
    config=rf_params
)

rf = RandomForestClassifier(**rf_params)
rf.fit(X_train_sc, y_train)

y_prob_rf = rf.predict_proba(X_test_sc)[:, 1]
y_pred_rf = rf.predict(X_test_sc)

log_model_metrics(run_rf, 'RandomForest', y_test, y_pred_rf, y_prob_rf, FEATURES, rf.feature_importances_)

run_rf.finish()
print("\nRandom Forest run complete!")

## 7. Model Comparison Summary

In [ ]:
results = pd.DataFrame({
    'Model': ['XGBoost', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_rf)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_prob_xgb),
        roc_auc_score(y_test, y_prob_rf)
    ],
    'Avg Precision': [
        average_precision_score(y_test, y_prob_xgb),
        average_precision_score(y_test, y_prob_rf)
    ]
})

print("\n=== Final Comparison ===")
print(results.to_string(index=False))

In [ ]:
# Bar chart comparison
metrics = ['Accuracy', 'ROC-AUC', 'Avg Precision']
x       = np.arange(len(metrics))
width   = 0.3

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, results.loc[results.Model=='XGBoost', metrics].values[0],
       width, label='XGBoost', color='steelblue')
ax.bar(x + width/2, results.loc[results.Model=='Random Forest', metrics].values[0],
       width, label='Random Forest', color='coral')

ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0.5, 1.0)
ax.set_title('Model Comparison — Wine Quality Classification')
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120)
plt.show()
print("Comparison chart saved.")

## 8. W&B Hyperparameter Sweep for XGBoost

This sweep searches over `eta` and `max_depth` to find the best configuration.

In [ ]:
sweep_config = {
    'method': 'grid',
    'metric': {'name': 'roc_auc', 'goal': 'maximize'},
    'parameters': {
        'eta':       {'values': [0.05, 0.1, 0.2]},
        'max_depth': {'values': [4, 6, 8]},
    }
}

def sweep_train():
    with wandb.init() as run:
        cfg = run.config
        params = {
            'objective':  'binary:logistic',
            'eta':        cfg.eta,
            'max_depth':  cfg.max_depth,
            'eval_metric': 'logloss',
            'seed': 42
        }
        dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
        dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=FEATURES)

        model = xgb.train(params, dtrain, num_boost_round=80,
                          evals=[(dtest, 'eval')], verbose_eval=False)
        y_prob = model.predict(dtest)
        auc    = roc_auc_score(y_test, y_prob)
        acc    = accuracy_score(y_test, (y_prob >= 0.5).astype(int))

        run.log({'roc_auc': auc, 'accuracy': acc})

sweep_id = wandb.sweep(sweep_config, project="Lab1-wine-quality-mlops")
wandb.agent(sweep_id, function=sweep_train, count=9)  # 3x3 grid

print("Sweep complete! View results at https://wandb.ai")